# Representações de Grafos: Matriz de Adjacência e Lista de Adjacência

Este notebook implementa e demonstra duas representações clássicas de grafos em estruturas de dados: matriz de adjacência e lista de adjacência. Inclui implementações em Python, conversões, visualização com networkx e benchmarks simples.

Objetivos:
- Implementar classes que representam arestas e grafos (matriz e lista).
- Demonstrar operações básicas: inserir, remover, verificar existência e iterar adjacências.
- Converter entre representações e comparar desempenho.

Referências: estruturas clássicas de grafos e exemplos didáticos.

## Índice

- [Introdução e definições](#Introducao-e-definicoes)
- [Terminologia essencial](#Terminologia-essencial)
- [Representações (visão geral)](#Representacoes-(visao-geral))
- [Matriz de Adjacência](#Matriz-de-Adjacencia)
- [Lista de Adjacência](#Lista-de-Adjacencia)
- [Conversões](#Conversoes)
- [Visualização](#Visualizacao)
- [Benchmark e complexidade](#Benchmark-e-complexidade)
- [Testes e referências](#Testes-e-referencias)

## Introdução e definições

Um grafo G = (V, E) é composto de vértices V e arestas E. Neste notebook focamos em representações em estruturas de dados (matriz de adjacência e lista de adjacência) e suas propriedades práticas.

## Terminologia essencial

- Vértice (nó)
- Aresta
- Grafo dirigido (DiGraph) vs não dirigido
- Grau, in-degree, out-degree
- Matriz de adjacência vs Lista de adjacência

Explicaremos as duas representações e mostraremos código que implementa cada uma.

## Representações com NetworkX

Mostraremos como converter nossas representações (matriz e lista) para objetos `networkx.Graph` / `DiGraph` e realizar operações básicas: listar nós, arestas e obter a matriz de adjacência (se o numpy estiver disponível). Isso ajuda a comparar a representação manual com as facilidades do NetworkX.

In [69]:
# Funções auxiliares para converter as representações para networkx
# Esta célula foi corrigida para não depender de nomes globais que
# possam não existir no momento da execução. Recebe explicitamente
# uma instância de GrafoLista ou GrafoMatriz.

import networkx as nx

def grafo_lista_para_networkx(gl, directed: bool = True):
    """Converte uma instância de GrafoLista para um objeto networkx."""
    if nx is None:
        print('networkx não disponível')
        return None
    G = nx.DiGraph() if directed else nx.Graph()
    # Adiciona nós explicitamente para preservar vértices isolados
    for i in range(gl.num_vertices):
        G.add_node(i)
    for i in range(gl.num_vertices):
        for a in gl.listas[i]:
            G.add_edge(a.v1, a.v2, weight=a.peso)
    return G

def grafo_matriz_para_networkx(gm, directed: bool = True):
    """Converte uma instância de GrafoMatriz para networkx via matriz_para_lista."""
    if nx is None:
        print('networkx não disponível')
        return None
    # Importante: matriz_para_lista deve estar definida no escopo global
    # onde esta célula é executada (ou no módulo).
    gl = matriz_para_lista(gm)
    return grafo_lista_para_networkx(gl, directed=directed)

# Exemplo seguro: se quiser testar aqui, passe explicitamente a instância
# GL ou crie uma nova. Não se apoie em nome GL estar definido fora.
if nx is not None:
    try:
        # se existir uma instância chamada GL no escopo, usamos; caso contrário, ignoramos
        G_nx = grafo_lista_para_networkx(GL)
        if G_nx is not None:
            print('Nodes (networkx):', list(G_nx.nodes()))
            print('Edges (networkx):', list(G_nx.edges()))
    except NameError:
        # GL não está definido aqui — isso é esperado em execuções parciais
        pass
else:
    print('networkx não instalado; para visualização, instale networkx e matplotlib')

Nodes (networkx): [0, 1, 2, 3, 4]
Edges (networkx): [(0, 2), (1, 2), (3, 4)]


## 1) Configuração do ambiente

Instale dependências necessárias (opcionais para visualização/benchmark):

- networkx, matplotlib para visualização
- memory_profiler (opcional) para medir memória

Instruções (executar no terminal do VS Code):

```bash
pip install networkx matplotlib pytest memory-profiler
```

Importações usadas neste notebook:

In [70]:
# Importações comuns
from dataclasses import dataclass
from typing import Optional, List, Iterator, Any, Tuple
import timeit

# Importações opcionais para visualização
try:
    import networkx as nx
    import matplotlib.pyplot as plt
except Exception:
    nx = None
    plt = None

print(f"Python version: {'.'.join(map(str, __import__('sys').version_info[:3]))}")

Python version: 3.12.10


## 2) Definição da classe Aresta

Usaremos uma dataclass para representar uma aresta com os campos `v1`, `v2` e `peso`.
Ela inclui um __repr__ legível para depuração.

In [71]:
@dataclass
class Aresta:
    v1: int
    v2: int
    peso: float = 1.0

    def __repr__(self) -> str:
        return f"Aresta({self.v1} -> {self.v2}, peso={self.peso})"

## 3) Implementação: Matriz de Adjacência (GrafoMatriz)

Classe que representa o grafo usando uma matriz de tamanho num_vertices x num_vertices.
- valores ausentes: None
- pos: vetor auxiliar para iteração nas adjacências (primeiroListaAdj/proxAdj)

Métodos implementados: insereAresta, existeAresta, retiraAresta, imprime, primeiroListaAdj, proxAdj.

In [72]:
class GrafoMatriz:
    def __init__(self, num_vertices: int):
        self.num_vertices = num_vertices
        self.mat: List[List[Optional[float]]] = [[None] * num_vertices for _ in range(num_vertices)]
        # posição usada para iteração em adjacências
        self._pos: List[int] = [-1] * num_vertices

    def insereAresta(self, v1: int, v2: int, peso: float = 1.0) -> None:
        self._check_vertex(v1)
        self._check_vertex(v2)
        self.mat[v1][v2] = peso

    def existeAresta(self, v1: int, v2: int) -> bool:
        self._check_vertex(v1)
        self._check_vertex(v2)
        return self.mat[v1][v2] is not None

    def retiraAresta(self, v1: int, v2: int) -> Optional[float]:
        self._check_vertex(v1)
        self._check_vertex(v2)
        prev = self.mat[v1][v2]
        self.mat[v1][v2] = None
        return prev

    def imprime(self) -> None:
        print("Matriz de adjacência (None = sem aresta):")
        for i in range(self.num_vertices):
            print([self.mat[i][j] for j in range(self.num_vertices)])

    def primeiroListaAdj(self, v: int) -> Optional[Aresta]:
        self._check_vertex(v)
        for j in range(self.num_vertices):
            if self.mat[v][j] is not None:
                self._pos[v] = j
                return Aresta(v, j, self.mat[v][j])
        self._pos[v] = -1
        return None

    def proxAdj(self, v: int) -> Optional[Aresta]:
        self._check_vertex(v)
        start = self._pos[v]
        if start == -1:
            return None
        for j in range(start + 1, self.num_vertices):
            if self.mat[v][j] is not None:
                self._pos[v] = j
                return Aresta(v, j, self.mat[v][j])
        self._pos[v] = -1
        return None

    def _check_vertex(self, v: int) -> None:
        if not (0 <= v < self.num_vertices):
            raise IndexError(f"Vértice {v} fora do intervalo [0, {self.num_vertices})")

## 4) Demonstração e testes: Matriz de Adjacência

Criamos um pequeno grafo, inserimos arestas, verificamos existência, removemos e iteramos adjacências.

In [73]:
# Demonstração: criar grafo com 5 vértices
G = GrafoMatriz(5)
G.insereAresta(0, 1, 2.5)
G.insereAresta(0, 2, 1.0)
G.insereAresta(1, 2, 3.0)
G.insereAresta(3, 4, 4.2)

assert G.existeAresta(0,1)
assert not G.existeAresta(2,0)

print('Adjacências do vértice 0:')
a = G.primeiroListaAdj(0)
while a is not None:
    print(' ', a)
    a = G.proxAdj(0)

print('\nImprimindo matriz:')
G.imprime()

# testar remoção
ant = G.retiraAresta(0,1)
print('\nRemovi aresta 0->1 com peso', ant)
assert not G.existeAresta(0,1)

Adjacências do vértice 0:
  Aresta(0 -> 1, peso=2.5)
  Aresta(0 -> 2, peso=1.0)

Imprimindo matriz:
Matriz de adjacência (None = sem aresta):
[None, 2.5, 1.0, None, None]
[None, None, 3.0, None, None]
[None, None, None, None, None]
[None, None, None, None, 4.2]
[None, None, None, None, None]

Removi aresta 0->1 com peso 2.5


## 5) Estruturas auxiliares: Celula e Lista

Implementaremos uma lista ligada simples para armazenar arestas em cada vértice.

In [74]:
class Celula:
    def __init__(self, dado: Aresta, prox: Optional['Celula'] = None):
        self.dado = dado
        self.prox = prox

    def __repr__(self):
        return f"Celula({self.dado})"

class Lista:
    def __init__(self):
        self.head: Optional[Celula] = None
        self._len = 0

    def insere(self, dado: Aresta) -> None:
        self.head = Celula(dado, self.head)
        self._len += 1

    def vazia(self) -> bool:
        return self.head is None

    def retira(self, v2: int) -> Optional[Aresta]:
        prev = None
        cur = self.head
        while cur is not None:
            if cur.dado.v2 == v2:
                # remove
                if prev is None:
                    self.head = cur.prox
                else:
                    prev.prox = cur.prox
                self._len -= 1
                return cur.dado
            prev = cur
            cur = cur.prox
        return None

    def __iter__(self) -> Iterator[Aresta]:
        cur = self.head
        while cur is not None:
            yield cur.dado
            cur = cur.prox

    def __len__(self) -> int:
        return self._len

    def __repr__(self) -> str:
        return f"Lista([{', '.join(repr(x) for x in self)}])"

## 6) Implementação: Lista de Adjacência (GrafoLista)

Cada vértice possui uma `Lista` de `Aresta` indicando seus vizinhos. Implementaremos os métodos solicitados.

In [75]:
class GrafoLista:
    def __init__(self, num_vertices: int):
        self.num_vertices = num_vertices
        self.listas: List[Lista] = [Lista() for _ in range(num_vertices)]
        self._iter_pos: List[Optional[Celula]] = [None] * num_vertices

    def insereAresta(self, v1: int, v2: int, peso: float = 1.0) -> None:
        self._check_vertex(v1)
        self._check_vertex(v2)
        self.listas[v1].insere(Aresta(v1, v2, peso))

    def existeAresta(self, v1: int, v2: int) -> bool:
        self._check_vertex(v1)
        self._check_vertex(v2)
        for a in self.listas[v1]:
            if a.v2 == v2:
                return True
        return False

    def listaAdjVazia(self, v: int) -> bool:
        self._check_vertex(v)
        return self.listas[v].vazia()

    def primeiroListaAdj(self, v: int) -> Optional[Aresta]:
        self._check_vertex(v)
        cell = self.listas[v].head
        self._iter_pos[v] = cell
        return cell.dado if cell is not None else None

    def proxAdj(self, v: int) -> Optional[Aresta]:
        self._check_vertex(v)
        cur = self._iter_pos[v]
        if cur is None:
            return None
        cur = cur.prox
        self._iter_pos[v] = cur
        return cur.dado if cur is not None else None

    def retiraAresta(self, v1: int, v2: int) -> Optional[Aresta]:
        self._check_vertex(v1)
        self._check_vertex(v2)
        return self.listas[v1].retira(v2)

    def imprime(self) -> None:
        print('Listas de adjacência:')
        for i, l in enumerate(self.listas):
            print(i, ':', l)

    def _check_vertex(self, v: int) -> None:
        if not (0 <= v < self.num_vertices):
            raise IndexError(f"Vértice {v} fora do intervalo [0, {self.num_vertices})")

## 7) Demonstração e testes: Lista de Adjacência

Construiremos o mesmo grafo do exemplo da matriz e validaremos operações equivalentes.

In [76]:
GL = GrafoLista(5)
GL.insereAresta(0, 1, 2.5)
GL.insereAresta(0, 2, 1.0)
GL.insereAresta(1, 2, 3.0)
GL.insereAresta(3, 4, 4.2)

assert GL.existeAresta(0,1)
assert not GL.existeAresta(2,0)

print('Adjacências do vértice 0 (lista):')
a = GL.primeiroListaAdj(0)
while a is not None:
    print(' ', a)
    a = GL.proxAdj(0)

print('\nImprimindo listas:')
GL.imprime()

ret = GL.retiraAresta(0,1)
print('\nRetirei', ret)
assert not GL.existeAresta(0,1)

Adjacências do vértice 0 (lista):
  Aresta(0 -> 2, peso=1.0)
  Aresta(0 -> 1, peso=2.5)

Imprimindo listas:
Listas de adjacência:
0 : Lista([Aresta(0 -> 2, peso=1.0), Aresta(0 -> 1, peso=2.5)])
1 : Lista([Aresta(1 -> 2, peso=3.0)])
2 : Lista([])
3 : Lista([Aresta(3 -> 4, peso=4.2)])
4 : Lista([])

Retirei Aresta(0 -> 1, peso=2.5)


## 8) Conversão entre representações

Funções para converter de GrafoMatriz para GrafoLista e vice-versa, e verificar equivalência das arestas.

In [77]:
def matriz_para_lista(gm: GrafoMatriz) -> GrafoLista:
    gl = GrafoLista(gm.num_vertices)
    for i in range(gm.num_vertices):
        for j in range(gm.num_vertices):
            if gm.mat[i][j] is not None:
                gl.insereAresta(i, j, gm.mat[i][j])
    return gl


def lista_para_matriz(gl: GrafoLista) -> GrafoMatriz:
    gm = GrafoMatriz(gl.num_vertices)
    for i in range(gl.num_vertices):
        for a in gl.listas[i]:
            gm.insereAresta(a.v1, a.v2, a.peso)
    return gm


def equivalentes(gm: GrafoMatriz, gl: GrafoLista) -> bool:
    if gm.num_vertices != gl.num_vertices:
        return False
    for i in range(gm.num_vertices):
        for j in range(gm.num_vertices):
            mat_val = gm.mat[i][j]
            # check in list
            found = False
            for a in gl.listas[i]:
                if a.v2 == j and a.peso == mat_val:
                    found = True
                    break
            if (mat_val is None and found) or (mat_val is not None and not found):
                return False
    return True

## 9) Visualização com networkx

Função utilitária para desenhar um GrafoLista ou GrafoMatriz usando networkx e matplotlib. Se o networkx não estiver instalado, pula a visualização.

In [78]:
def desenha_grafo_de_lista(gl: GrafoLista, directed: bool = True):
    if nx is None:
        print('networkx não instalado; instale networkx e matplotlib para visualizar')
        return
    G = nx.DiGraph() if directed else nx.Graph()
    for i in range(gl.num_vertices):
        G.add_node(i)
    for i in range(gl.num_vertices):
        for a in gl.listas[i]:
            G.add_edge(a.v1, a.v2, weight=a.peso)
    pos = nx.spring_layout(G)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw(G, pos, with_labels=True, node_color='lightblue', arrows=directed)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.show()

# Helper to draw from matrix
def desenha_grafo_de_matriz(gm: GrafoMatriz, directed: bool = True):
    gl = matriz_para_lista(gm)
    desenha_grafo_de_lista(gl, directed=directed)

## 10) Comparação de complexidade e benchmark

Mediremos tempo de inserção em grafos densos e esparsos para comparar o comportamento prático (exemplo simples usando timeit).

In [79]:
def benchmark_insercao(n: int, densidade: float = 0.1):
    # densidade = fração de possíveis arestas que serão inseridas
    gm = GrafoMatriz(n)
    gl = GrafoLista(n)
    edges = []
    import random
    for i in range(n):
        for j in range(n):
            if i != j and random.random() < densidade:
                edges.append((i,j, random.random()))

    def insere_matriz():
        for (i,j,w) in edges:
            gm.insereAresta(i,j,w)

    def insere_lista():
        for (i,j,w) in edges:
            gl.insereAresta(i,j,w)

    t_mat = timeit.timeit(insere_matriz, number=1)
    t_list = timeit.timeit(insere_lista, number=1)
    return t_mat, t_list

# Exemplo rápido (atenção: pode demorar para n grande)
print('Benchmark rápido n=200, densidade=0.01:')
print(benchmark_insercao(200, 0.01))

Benchmark rápido n=200, densidade=0.01:
(0.0002536000101827085, 0.0006136000156402588)


## 11) Testes unitários com pytest (exemplo)

A seguir um esqueleto de testes que pode ser salvo em `test_grafos.py` para rodar com pytest.

In [80]:
# Conteúdo sugerido para test_grafos.py
test_py = '''
from representacoes_grafos import Aresta, GrafoMatriz, GrafoLista, matriz_para_lista, lista_para_matriz

def test_matriz_lista_equivalencia():
    gm = GrafoMatriz(4)
    gm.insereAresta(0,1,1.0)
    gm.insereAresta(1,2,2.0)
    gl = matriz_para_lista(gm)
    assert gl.existeAresta(0,1)
    gm2 = lista_para_matriz(gl)
    assert gm2.existeAresta(1,2)
'''
print('Exemplo de testes (salve em test_grafos.py):')
print(test_py)

Exemplo de testes (salve em test_grafos.py):

from representacoes_grafos import Aresta, GrafoMatriz, GrafoLista, matriz_para_lista, lista_para_matriz

def test_matriz_lista_equivalencia():
    gm = GrafoMatriz(4)
    gm.insereAresta(0,1,1.0)
    gm.insereAresta(1,2,2.0)
    gl = matriz_para_lista(gm)
    assert gl.existeAresta(0,1)
    gm2 = lista_para_matriz(gl)
    assert gm2.existeAresta(1,2)

